# 05 · BGE-small student: alignment + contrastive refinement, full fine-tune (DDP, 2 GPUs)
A second, much smaller student: `BAAI/bge-small-en-v1.5` (BERT, 12 layers, hidden 384, 33M params), CLS pooling,
right padding. A linear head 384→256 maps it into the teacher index space. **Every weight trains** (no LoRA, no
pruning), in two stages:
1. **Align**: put the student's query vector on the teacher's vector of the same query,
   L2 = ||s − t||² = 2 − 2·cos (as in `02`).
2. **Contrastive refinement**: from stage 1, rank the labelled positive above the batch's other teacher docs
   (as in `03`): L = CE(logits / tau, own positive) + l2_weight · L2. Documents stay the frozen teacher `corpus.npy`.

Each stage is its own checkpoint dir (`bge_aligned/`, `bge_aligned_cr/`, each with `final/`) and its own row.
Re-running resumes and a finished stage is skipped.

Input: `01`. Output (attach to `04`): `runs/<mode>/{bge_aligned, bge_aligned_cr}/`, `runs/<mode>/metrics.json`.

In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [2]:
# setup: clone/pull the repo on Kaggle, put it on sys.path, install what's missing (re-run to pull latest code)
import os, sys, importlib.util
REPO_URL = "https://github.com/pyther-hub/hlm-poc.git"
ON_KAGGLE = os.path.exists("/kaggle/working")
REPO = "/kaggle/working/hlm-poc" if ON_KAGGLE else os.path.abspath("..")
if ON_KAGGLE:
    if not os.path.exists(REPO):
        !git clone -q {REPO_URL} {REPO}
    else:
        !git -C {REPO} pull -q
sys.path.insert(0, REPO)

# HF token (higher rate limits, faster downloads). Kaggle: Add-ons -> Secrets -> add "HF_TOKEN", attach to notebook.
# Spawned workers inherit os.environ. Locally: export HF_TOKEN or `huggingface-cli login`.
if ON_KAGGLE and "HF_TOKEN" not in os.environ:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print(f"HF_TOKEN secret not found ({type(e).__name__}): downloads stay unauthenticated")

import transformers
from packaging.version import Version
if Version(transformers.__version__) < Version("4.56"):  # Qwen3 + the dtype= kwarg
    !pip install -q -U "transformers>=4.56"
    print("transformers upgraded: restart the kernel, then re-run this cell")
for pkg in ("datasets", "pyarrow", "accelerate"):
    if importlib.util.find_spec(pkg) is None:
        !pip install -q {pkg}



## Control panel

In [3]:
import os
import shutil
import torch
from model import get_device
from teacher import TEACHER_DEFAULTS as T

DEBUG = False    # use the same value as in 01
NPROCS = 2

# --- paths. On Kaggle, earlier notebooks' outputs are attached as inputs (read only); set the slugs to yours.
#     Off Kaggle everything lives under the repo, so every IN_* is the same dir.
ROOT = "/kaggle/working" if ON_KAGGLE else REPO
MODE = "debug" if DEBUG else "full"
KAGGLE_IN = "/kaggle/input/datasets/tensorpanda231"
IN_01 = f"{KAGGLE_IN}/corpus-teacher-embd" if ON_KAGGLE else REPO
DATA_PATH = f"{IN_01}/cache/{MODE}/data.json"
EMB_DIR = f"{IN_01}/cache/{MODE}/emb/teacher"
RUN_DIR = f"{ROOT}/runs/{MODE}"
PREV_METRICS = f"{IN_01}/runs/{MODE}/metrics.json"

STUDENT = "BAAI/bge-large-en-v1.5"
# BGE's optional retrieval instruction is "Represent this sentence for searching relevant passages: "; off by
# default to match the other students. Must be the same in ALIGN, REFINE and EVAL.
QUERY_PREFIX = ""

# --- stage 1: alignment, full fine-tune (align.ALIGN_DEFAULTS; unknown keys raise)
ALIGN = dict(model=STUDENT,
             full=True,                           # every base weight + head trains; no LoRA (lora_* unused)
             dim=T["dim"],                        # head output = teacher index dim
             query_prefix=QUERY_PREFIX, max_query_len=32,
             loss="l2", loss_kwargs={},           # ||s - t||^2 = 2 - 2 cos between unit vectors
             lr=1e-4,                             # all base weights; the usual full-FT range for a 33M BERT is 2e-5..1e-4
             head_lr=3e-4,                        # the head starts random
             weight_decay=0.0, warmup_ratio=0.05,
             epochs=8, batch_size=128,            # per GPU; global batch = batch_size * NPROCS
             grad_clip=1.0, amp=True, seed=0,
             log_every=20, eval_every=100, save_every=400, keep_last=2,
             data_path=DATA_PATH, teacher_emb_dir=EMB_DIR, out_dir=f"{RUN_DIR}/bge_aligned")

# --- stage 2: contrastive refinement of every weight (refine.REFINE_DEFAULTS; init_adapter / out_dir set below)
REFINE = dict(dim=T["dim"], query_prefix=QUERY_PREFIX, max_query_len=32,
              tau=0.05,                            # temperature
              margin=0.1,                          # mask negatives with s > s(q, d+) + margin (likely false negatives)
              n_neg=1,                             # negatives per query: 1 = stored one, k > 1 adds k-1 teacher-mined
              l2_weight=0.1,                       # keeps the student near the teacher query space; 0 = pure L_CR
              lr=3e-5, head_lr=3e-5, weight_decay=0.05, warmup_ratio=0.05,
              epochs=8, batch_size=128,            # queries per GPU; docs per step = (1 + n_neg) * batch_size * NPROCS
              grad_clip=1.0, amp=True, seed=0,
              log_every=20, eval_every=100, save_every=500, keep_last=2,
              data_path=DATA_PATH, teacher_emb_dir=EMB_DIR)

# --- evaluation (evaluation.evaluate_model kwargs): student queries vs the frozen teacher corpus, top 100
EVAL = dict(splits=("val", "test"), dim=T["dim"], max_query_len=32, query_prefix=QUERY_PREFIX, device=get_device(),
            dtype="fp16")
LATENCY = dict(n_queries=2000, batch_sizes=(1, 32))

if DEBUG:
    for cfg in (ALIGN, REFINE):
        cfg.update(log_every=5, eval_every=20, save_every=20)
    LATENCY.update(n_queries=200)

## 1 · Check inputs
Teacher vectors valid (finite, unit norm, `data.json` order) for the corpus, val/test and the alignment queries.

In [4]:
from data import load_emb, load_json
from evaluation import check_emb, load_index

# rows accumulate across notebooks: start from the previous notebook's metrics.json (copied once)
os.makedirs(RUN_DIR, exist_ok=True)
if not os.path.exists(f"{RUN_DIR}/metrics.json") and os.path.exists(PREV_METRICS):
    shutil.copy(PREV_METRICS, f"{RUN_DIR}/metrics.json")
data = load_json(DATA_PATH)
index = load_index(data, EMB_DIR, ("val", "test"))   # (teacher docs, {split: teacher queries})
emb, ids = load_emb(EMB_DIR, "align")
check_emb(emb, ids, data["align"]["ids"], "align")
assert set(data["train"]["ids"]) <= set(data["align"]["ids"]), "train queries need teacher vectors (align set)"
print(f"align {emb.shape}: {len(ids) // (ALIGN['batch_size'] * NPROCS)} align steps, "
      f"{len(data['train']['ids']) // (REFINE['batch_size'] * NPROCS)} refine steps")

[eval] index ok: corpus (299985, 256), val (5000, 256), test (5000, 256)
align (200000, 256): 781 align steps, 351 refine steps


## 2 · Align (full fine-tune)
Rank r takes its slice of every global batch; DDP averages the gradients of every weight. Val L2 / cos / nDCG@10 are
logged every `eval_every` steps.

In [5]:
from align import align_main
from run_exp import train_once

runs = {"bge_aligned": train_once(align_main, ALIGN, NPROCS)}

[rank0] joined process group on cuda:0 (14.5/14.6 GB free)[rank1] joined process group on cuda:1 (14.5/14.6 GB free)

[rank0] loading BAAI/bge-large-en-v1.5[rank1] waiting for rank 0 to load the model



Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2485.66it/s]


[rank0] model loaded on cuda:0 (13.2/14.6 GB free)


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1030 21:40:41.824777263 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


[rank1] loading BAAI/bge-large-en-v1.5
[align] 200000 examples, global batch 256, 781 steps/epoch, 6248 steps, 334.35M trainable params
[rank0] wrapping in DDP (13.1/14.6 GB free)


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2580.03it/s]


[rank1] model loaded on cuda:1 (13.1/14.6 GB free)
[rank1] wrapping in DDP (13.1/14.6 GB free)
[rank0] training from step 0 of 6248
[rank1] training from step 0 of 6248


/kaggle/working/hlm-poc/align.py:214: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  means = {("train_loss" if k == "loss" else k): reduce_mean(float(v), device) for k, v in parts.items()}
/kaggle/working/hlm-poc/align.py:214: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  means = {("train_loss" if k == "loss" else k): reduce_mean(float(v), device) for k, v in parts.items()}


[align] step 20  epoch 0  train_loss 1.736  lr 6.731e-06  min 0.2068
[align] step 40  epoch 0  train_loss 1.349  lr 1.314e-05  min 0.3879
[align] step 60  epoch 0  train_loss 1.179  lr 1.955e-05  min 0.5729
[align] step 80  epoch 0  train_loss 1.006  lr 2.596e-05  min 0.755
[align] step 100  epoch 0  train_loss 0.8656  lr 3.237e-05  min 0.9441
[rank1] step 100: waiting for rank 0 to evaluate
[align] step 100  val_loss 0.8509  val_ndcg@10 0.03395  val_recall@10 0.06528  val_cos 0.5746
[align] step 120  epoch 0  train_loss 0.8163  lr 3.878e-05  min 1.217
[align] step 140  epoch 0  train_loss 0.7339  lr 4.519e-05  min 1.407
[align] step 160  epoch 0  train_loss 0.7009  lr 5.16e-05  min 1.605
[align] step 180  epoch 0  train_loss 0.6546  lr 5.801e-05  min 1.803
[rank1] step 200: waiting for rank 0 to evaluate
[align] step 200  epoch 0  train_loss 0.6123  lr 6.442e-05  min 1.996
[align] step 200  val_loss 0.5782  val_ndcg@10 0.3339  val_recall@10 0.4988  val_cos 0.7109
[align] step 220  epo

## 3 · Contrastive refinement from the aligned student, compare before / after
`adapter_config.json` has `full: true`, so refinement trains every weight. Trains `bge_aligned_cr` if missing,
evaluates the missing rows (val + test), prints the gain and the table.

In [6]:
import matplotlib.pyplot as plt
from IPython.display import display
from run_exp import exp_cr

res = exp_cr(REFINE, runs, data, index, RUN_DIR, EVAL, NPROCS)
runs["bge_aligned_cr"] = f"{RUN_DIR}/bge_aligned_cr/final"
for fig in res["figs"]:
    display(fig)
plt.close("all")

[rank1] joined process group on cuda:1 (14.5/14.6 GB free)
[rank0] joined process group on cuda:0 (14.5/14.6 GB free)
[rank0] loading /kaggle/working/runs/full/bge_aligned/final
[rank1] waiting for rank 0 to load the model


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 2322.26it/s]


[rank0] model loaded on cuda:0 (12.0/14.6 GB free)


/usr/local/lib/python3.13/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W1030 22:46:31.421798028 ProcessGroupNCCL.cpp:5188] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()


[rank1] loading /kaggle/working/runs/full/bge_aligned/final
[rank0] corpus on cuda:0 (11.7/14.6 GB free), mining negatives
[refine] 90000 examples, global batch 256, 351 steps/epoch, 2808 steps, 334.35M trainable params
[rank0] wrapping in DDP (11.4/14.6 GB free)


...

[rank1] model loaded on cuda:1 (11.8/14.6 GB free)
[rank1] corpus on cuda:1 (11.7/14.6 GB free), mining negatives
[rank1] wrapping in DDP (11.4/14.6 GB free)
[rank0] training from step 0 of 2808
[rank1] training from step 0 of 2808


/kaggle/working/hlm-poc/align.py:214: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  means = {("train_loss" if k == "loss" else k): reduce_mean(float(v), device) for k, v in parts.items()}
/kaggle/working/hlm-poc/align.py:214: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  means = {("train_loss" if k == "loss" else k): reduce_mean(float(v), device) for k, v in parts.items()}


[refine] step 20  epoch 0  train_loss 0.1684  cr 0.1531  l2 0.1536  masked 1.225e-05  lr 4.5e-06  min 0.2188
[refine] step 40  epoch 0  train_loss 0.1761  cr 0.1591  l2 0.1698  masked 0  lr 8.786e-06  min 0.4247
[refine] step 60  epoch 0  train_loss 0.1549  cr 0.1373  l2 0.1756  masked 6.123e-06  lr 1.307e-05  min 0.6274
[refine] step 80  epoch 0  train_loss 0.1101  cr 0.09204  l2 0.1802  masked 6.123e-06  lr 1.736e-05  min 0.8238
[refine] step 100  epoch 0  train_loss 0.1983  cr 0.1797  l2 0.1855  masked 0.0002143  lr 2.164e-05  min 1.015
[rank1] step 100: waiting for rank 0 to evaluate
[refine] step 100  val_loss 0.2168  val_ndcg@10 0.7491  val_recall@10 0.9125  val_cos 0.8916
[refine] step 120  epoch 0  train_loss 0.1991  cr 0.1806  l2 0.1851  masked 3.061e-05  lr 2.593e-05  min 1.278
[refine] step 140  epoch 0  train_loss 0.1552  cr 0.1352  l2 0.1999  masked 6.123e-06  lr 3e-05  min 1.464
[refine] step 160  epoch 0  train_loss 0.1802  cr 0.1603  l2 0.1996  masked 6.123e-06  lr 3e-0

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1004, in save
    _save(
    ~~~~~^
        obj,
        ^^^^
    ...<3 lines>...
        _disable_byteorder_record,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1313, in _save
    zip_file.write_record(name, storage, num_bytes)
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: basic_ios::clear: iostream error

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/kaggle/working/hlm-poc/refine.py", line 93, in refine_main
    train(rank, world_size, cfg, device)
    ~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/hlm-poc/refine.py", line 127, in train
    run(model, tokenizer, adapter_cfg, cfg, rank, world_size, device, len(queries["ids"]), batch_loss, data, "refine")
    ~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

ProcessExitedException: process 1 terminated with signal SIGABRT

## 4 · Training curves
`val_loss` is L2 in both stages (comparable); `train_loss` is L2 in stage 1, CR + l2_weight · L2 in stage 2.

In [ ]:
from run_exp import plot_curves, save_fig

fig = plot_curves({n: f"{RUN_DIR}/{n}" for n in runs}, ("train_loss", "val_loss", "val_ndcg@10"), "bge-small")
display(save_fig(fig, RUN_DIR, "bge_small_curves"))
plt.close("all")

## 5 · Retrieval quality (val + test)
Student queries vs the frozen teacher corpus. retention = nDCG@10 / teacher nDCG@10.

In [ ]:
from run_exp import print_rows

metrics = load_json(f"{RUN_DIR}/metrics.json")
for split in ("val", "test"):
    print_rows(metrics, list(runs), split)
    print()

## 6 · Query latency (one GPU, fp16)

In [ ]:
from evaluation import measure_latency
from model import load_encoder
from teacher import free_gpu_memory

texts = data["val"]["texts"][:LATENCY["n_queries"]]
for name, path in runs.items():
    model, tok = load_encoder(path, EVAL["device"], torch.float16)
    print(f"--- {name}: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params")
    measure_latency(model, tok, texts, EVAL["device"], T["dim"], 32, QUERY_PREFIX, LATENCY["batch_sizes"])
    del model
    free_gpu_memory()